In [1]:
!pip install ultralytics -q
!pip install supervision -q
!pip install opencv-python -q
!pip install pandas -q

In [2]:
import os
print(os.getcwd())

os.chdir('/content/drive/MyDrive/Colab Notebooks/SoccerAnalysis')
print(os.getcwd())

/content
/content/drive/MyDrive/Colab Notebooks/SoccerAnalysis


In [ ]:
from utils import read_video, save_video
from tracker import Tracker
import cv2
from team_assigner import TeamAssigner
from player_ball_assigner import PlayerBallAssigner
def main():
    #read video
    video_frames=read_video(r"C:\Users\Kristal\Desktop\Colab_Notebooks\SoccerAnalysis\input_videos\08fd33_4.mp4")

    #Initialize tracker
    tracker=Tracker(r'C:\Users\Kristal\Desktop\Colab_Notebooks\SoccerAnalysis\model\best.pt')
    tracks =tracker.get_objects_tracks(video_frames,read_from_stub=True,stub_path='C:\Users\Kristal\Desktop\Colab_Notebooks\SoccerAnalysis\stubs\track_stubs.pkl')

    #interpolate Ball positions
    tracks['ball']=tracker.interpolate_ball_positions(tracks['ball'])
    #assign player teams
    team_assigner=TeamAssigner()
    team_assigner.assign_team_color(video_frames[0],tracks['players'][0])

    for frame_num,player_track in enumerate(tracks['players']):
      for player_id,track in player_track.items():
        team=team_assigner.get_player_team(video_frames[frame_num],track['bbox'],player_id)
        tracks['players'][frame_num][player_id]['team']=team
        tracks['players'][frame_num][player_id]['team_color']=team_assigner.team_colors[team]

    # assign ball acquisition
    player_assigner=PlayerBallAssigner()
    for frame_num,player_track in enumerate(tracks['players']):
      ball_bbox=tracks['ball'][frame_num][1]['bbox']
      assigned_player=player_assigner.assign_ball_to_player(player_track,ball_bbox)

      if assigned_player != -1:
        tracks['players'][frame_num][assigned_player]['has_ball']=True


    #draw output
    output_video_frames=tracker.draw_annotations(video_frames,tracks)
    #save video
    save_video(output_video_frames, r"C:\Users\Kristal\Desktop\Colab_Notebooks\SoccerAnalysis\output_videos/output_video(player_ball_assigner).avi")

if __name__ == "__main__":
    main()